# 01 — DataFrame Basics
SparkSession, DataFrame API, Spark SQL, Window Functions, CTEs.

In [1]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('spark-notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')
spark.range(3).show()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 07:07:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
W20260408 07:07:43.524490  1602 MemoryArbitrator.cpp:84] Query memory capacity[288.00MB] is set for NOOP arbitrator which has no capacity enforcement
26/04/08 07:07:44 WARN SparkShimProvider: Spark runtime version 4.0.2 is not matched with Gluten's fully tested version 4.0.1


Spark 4.0.2  |  Mode: Gluten/Velox


26/04/08 07:07:47 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=0], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(id#0L, Some(Etc/UTC)), class name: ToPrettyString.
[Stage 0:>                                                          (0 + 4) / 4]

+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



## DataFrame API

In [2]:
schema = StructType([
    StructField('id',         IntegerType()),
    StructField('name',       StringType()),
    StructField('department', StringType()),
    StructField('salary',     DoubleType()),
    StructField('country',    StringType()),
])
employees = spark.createDataFrame([
    (1,'Alice','Engineering',95000.0,'US'),
    (2,'Bob','Marketing',72000.0,'UK'),
    (3,'Carol','Engineering',88000.0,'US'),
    (4,'Dave','HR',65000.0,'DE'),
    (5,'Eve','Engineering',102000.0,'US'),
    (6,'Frank','Marketing',68000.0,'UK'),
    (7,'Grace','HR',71000.0,'DE'),
    (8,'Hank','Engineering',91000.0,'US'),
], schema)
employees.show()
employees.printSchema()


26/04/08 07:07:48 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=1], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(id#5, Some(Etc/UTC)), class name: ToPrettyString.
[Stage 2:===================>                                       (1 + 2) / 3]

+---+-----+-----------+--------+-------+
| id| name| department|  salary|country|
+---+-----+-----------+--------+-------+
|  1|Alice|Engineering| 95000.0|     US|
|  2|  Bob|  Marketing| 72000.0|     UK|
|  3|Carol|Engineering| 88000.0|     US|
|  4| Dave|         HR| 65000.0|     DE|
|  5|  Eve|Engineering|102000.0|     US|
|  6|Frank|  Marketing| 68000.0|     UK|
|  7|Grace|         HR| 71000.0|     DE|
|  8| Hank|Engineering| 91000.0|     US|
+---+-----+-----------+--------+-------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- country: string (nullable = true)



## Aggregations

In [3]:
employees.groupBy('department').agg(
    F.count('id').alias('headcount'),
    F.avg('salary').alias('avg_salary'),
    F.max('salary').alias('max_salary'),
).orderBy('avg_salary', ascending=False).show()


26/04/08 07:07:51 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=2], due to: [FallbackByBackendSettings] Validation failed on node Exchange
26/04/08 07:07:51 WARN GlutenFallbackReporter: Validation failed for plan: TakeOrderedAndProject[QueryId=2], due to: [FallbackByBackendSettings] Validation failed on node TakeOrderedAndProject


+-----------+---------+----------+----------+
| department|headcount|avg_salary|max_salary|
+-----------+---------+----------+----------+
|Engineering|        4|   94000.0|  102000.0|
|  Marketing|        2|   70000.0|   72000.0|
|         HR|        2|   68000.0|   71000.0|
+-----------+---------+----------+----------+



## Window Functions & CTEs

In [4]:
employees.createOrReplaceTempView('employees')
spark.sql("""
SELECT name, department, salary,
       RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rank,
       salary - AVG(salary) OVER (PARTITION BY department)       AS diff_from_avg
FROM employees
""").show()


26/04/08 07:07:52 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=4], due to: [FallbackByBackendSettings] Validation failed on node Exchange
26/04/08 07:07:52 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=4], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(name#6, Some(Etc/UTC)), class name: ToPrettyString.


+-----+-----------+--------+----+-------------+
| name| department|  salary|rank|diff_from_avg|
+-----+-----------+--------+----+-------------+
|  Eve|Engineering|102000.0|   1|       8000.0|
|Alice|Engineering| 95000.0|   2|       1000.0|
| Hank|Engineering| 91000.0|   3|      -3000.0|
|Carol|Engineering| 88000.0|   4|      -6000.0|
|Grace|         HR| 71000.0|   1|       3000.0|
| Dave|         HR| 65000.0|   2|      -3000.0|
|  Bob|  Marketing| 72000.0|   1|       2000.0|
|Frank|  Marketing| 68000.0|   2|      -2000.0|
+-----+-----------+--------+----+-------------+



In [5]:
spark.sql("""
WITH dept_stats AS (
    SELECT department, AVG(salary) AS avg_sal
    FROM employees GROUP BY department
)
SELECT e.name, e.department, e.salary,
       ROUND(e.salary / d.avg_sal * 100, 1) AS pct_of_avg
FROM employees e JOIN dept_stats d USING (department)
ORDER BY pct_of_avg DESC
""").show()


26/04/08 07:07:53 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=5], due to: [FallbackByBackendSettings] Validation failed on node Exchange
26/04/08 07:07:53 WARN GlutenFallbackReporter: Validation failed for plan: Exchange[QueryId=5], due to: [FallbackByBackendSettings] Validation failed on node Exchange


+-----+-----------+--------+----------+
| name| department|  salary|pct_of_avg|
+-----+-----------+--------+----------+
|  Eve|Engineering|102000.0|     108.5|
|Grace|         HR| 71000.0|     104.4|
|  Bob|  Marketing| 72000.0|     102.9|
|Alice|Engineering| 95000.0|     101.1|
|Frank|  Marketing| 68000.0|      97.1|
| Hank|Engineering| 91000.0|      96.8|
| Dave|         HR| 65000.0|      95.6|
|Carol|Engineering| 88000.0|      93.6|
+-----+-----------+--------+----------+



26/04/08 07:07:53 WARN GlutenFallbackReporter: Validation failed for plan: TakeOrderedAndProject[QueryId=5], due to: [FallbackByBackendSettings] Validation failed on node TakeOrderedAndProject
